# **Proyecto SQL**

Analizar la base de datos de un servicio de libros digitales con el fin de identificar patrones relevantes sobre:
- Publicación de libros
- Actividad de usuarios (calificaciones y reseñas)
- Desempeño de editoriales
- Autores mejor valorados

El análisis busca generar información que permita diseñar una propuesta de valor competitiva para un nuevo producto en el mercado de lectura digital.

In [1]:
import pandas as pd
from sqlalchemy import create_engine

db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-final-project-db'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode': 'require'})

engine.connect()

In [2]:

query = """
SELECT *
FROM books 
ORDER BY 
    publication_date DESC
    LIMIT 5;
"""
pd.io.sql.read_sql(query, con=engine)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,43,377,A Quick Bite (Argeneau #1),360,2020-03-31,28
1,635,166,The Art of Loving,192,2019-08-06,130
2,445,612,Monster,281,2019-03-05,14
3,293,80,Ham on Rye,288,2014-07-29,99
4,993,80,Women,291,2014-07-29,99


In [9]:
# Número de libros publicados después del 1 de enero del 2000
query = """
SELECT COUNT (*)
FROM 
    books
WHERE 
    publication_date > '2000-01-01';
"""

pd.io.sql.read_sql(query, con=engine)

,count
0,819


**Conclusión**

Se identificaron 819 libros publicados después del 1 de enero del 2000, lo que significa que una parte relevante de la base de datos corresponde a publicaciones relativmente recientes.

In [8]:
# Reseñas de usuarios y la calificación promedio para cada libro
query = """
SELECT 
    b.book_id,
    b.title,
    COUNT (DISTINCT r.review_id) AS num_reviews,
    AVG(rt.rating) AS avg_rating
FROM
    books b
    LEFT JOIN reviews r ON b.book_id = r.book_id
    LEFT JOIN ratings rt ON b.book_id = rt.book_id
GROUP BY 
    b.book_id, b.title
ORDER BY 
    num_reviews DESC;
"""


pd.io.sql.read_sql(query, con=engine)

,book_id,title,num_reviews,avg_rating
0,948,Twilight (Twilight #1),7,3.662500
1,963,Water for Elephants,6,3.977273
2,734,The Glass Castle,6,4.206897
3,302,Harry Potter and the Prisoner of Azkaban (Harr...,6,4.414634
4,695,The Curious Incident of the Dog in the Night-Time,6,4.081081
...,...,...,...,...
995,83,Anne Rice's The Vampire Lestat: A Graphic Novel,0,3.666667
996,808,The Natural Way to Draw,0,3.000000
997,672,The Cat in the Hat and Other Dr. Seuss Favorites,0,5.000000
998,221,Essential Tales and Poems,0,4.000000


**Conclusión**

Se analizó el número de reseñas de usuarios y la calificación promedio para cada libro. Para evitar duplicaciones causadas por las relaciones uno-a-muchos entre las tablas de reseñas y calificaciones, se utilizaron las funciones GROUP BY y conteos distintos (COUNT(DISTINCT)), asegurando que cada reseña se contabilizara correctamente.

Los resultados muestran que algunos títulos concentran el mayor número de reseñas, entre ellos Twilight, Water for Elephants, The Glass Castle, Harry Potter and the Prisoner of Azkaban y The Curious Incident of the Dog in the Night-Time. En general, estos libros mantienen calificaciones promedio relativamente altas, superiores a 3 puntos. Destaca especialmente Harry Potter and the Prisoner of Azkaban, que presenta una calificación cercana a 4.5.

En cuanto al contenido, los libros más reseñados no pertenecen a un único género; se observan obras de romance, fantasía, autobiografía y comedia. Esto sugiere que el alto nivel de interacción de los usuarios no está limitado a un solo tipo de literatura, sino que abarca diversos intereses entre los lectores.

In [10]:
# Editorial con mayor número de libros con más de 50 páginas 
query = """
SELECT 
    p.publisher,
    COUNT(b.book_id) AS num_books
FROM 
    books b
    JOIN publishers p ON b.publisher_id = p.publisher_id
WHERE 
    num_pages > 50
GROUP BY 
    p.
    publisher
ORDER BY
    num_books DESC
LIMIT 1;
"""

pd.io.sql.read_sql(query, con=engine)

,publisher,num_books
0,Penguin Books,42


**Conclusión**

A partir del análisis de la tabla de libros y editoriales, se identificó que Penguin Books es la editorial con el mayor número de publicaciones dentro del conjunto analizado, con 42 libros que superan las 50 páginas. Esto se puede deber a la gran presencia y popularidad, lo que aumenta la probabilidad de que una mayor cantidad de sus títulos aparezca en la base de datos.

In [11]:
# Autor con la calificación promedio más alta 
# Considerando solo los libros con al menos 50 calificaciones
query = """
SELECT
    a.author,
    AVG(r.rating) AS avg_rating,
    COUNT(r.rating_id) AS num_ratings
FROM 
    books b
    JOIN authors a ON b.author_id = a.author_id
    JOIN ratings r ON b.book_id = r.book_id
GROUP BY 
    a.author
HAVING COUNT(r.rating_id) >= 50
ORDER BY 
    avg_rating DESC
LIMIT 1;
"""

pd.io.sql.read_sql(query, con=engine)

,author,avg_rating,num_ratings
0,Diana Gabaldon,4.3,50


**Conclusión**

El análisis muestra que Diana Gabaldon es la autora con la calificación promedio más alta, con una puntuación promedio de 4.3 basada en 50 calificaciones.

Este resultado sugiere que los libros de esta autora mantienen una base de lectores leales y una alta satisfacción con sus obras.

In [13]:
# Número promedio de reseñas de texto
# Usuarios que calificaron más de 50 libros
query = """
SELECT 
    AVG(review_count) AS avg_reviews
FROM (
    SELECT 
        reviews.username,
        COUNT(reviews.review_id) AS review_count
    FROM reviews
    WHERE reviews.username IN (
        SELECT username
        FROM ratings
        GROUP BY username
        HAVING COUNT(rating_id) > 50
    )
    GROUP BY reviews.username
) AS user_reviews;
"""

pd.io.sql.read_sql(query, con=engine)

,avg_reviews
0,24.333333



El número promedio de reseñas de texto entre los usuarios que han calificado más de 50 libros es 24.33 reseñas.

Los usuarios que califican una gran cantidad de libros no necesariamente escriben reseñas para todos ellos. Esto puede deberse a que calificar un libro requiere menos esfuerzo y es más rápido que escribir una reseña.